In [ ]:
import random
from pathlib import Path

import numpy as np
import tensorflow as tf
from solve.static_hypernet import Solver

import matplotlib.pyplot as plt

from datasets import Mnist, Cifar10, SVHN, FashionMnist
from model.simple_cnn import SimpleCNN

In [ ]:
def set_random_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)


def build_run_paths(dataset, model, hyper_mode = False):
    run_name = f"{dataset}_{model}"
    if hyper_mode:
        run_name += "_hyper"
    run_root = Path("runs") / run_name
    return {
        "logpath": str(run_root / "logs"),
        "save_dir": str(run_root / "checkpoints"),
    }

set_random_seed(42)

In [ ]:
run_mnist_cnn_paths = build_run_paths("mnist", "simplecnn")


mnist_cnn_solver = Solver(
    dataset="mnist",
    model="simplecnn",
    max_epoch=20,
    hyper_mode=False,
    logpath=run_mnist_cnn_paths["logpath"],
    save_dir=run_mnist_cnn_paths["save_dir"],
    show_sample=True,
    show_filters=True,
)
mnist_cnn_solver.train()

In [ ]:
run_mnist_cnn_hyper_paths = build_run_paths("mnist", "simplecnn", hyper_mode=True)

mnist_cnn_hyper_solver = Solver(
    dataset="mnist",
    model="simplecnn",
    max_epoch=20,
    hyper_mode=True,
    logpath=run_mnist_cnn_hyper_paths["logpath"],
    save_dir=run_mnist_cnn_hyper_paths["save_dir"],
    show_sample=True,
    show_filters=True,
)
mnist_cnn_hyper_solver.train()

In [ ]:
run_fashion_mnist_cnn_paths = build_run_paths("fashion_mnist", "simplecnn")

fashion_mnist_cnn_solver = Solver(
    dataset="fashion_mnist",
    model="simplecnn",
    max_epoch=1,
    hyper_mode=False,
    logpath=run_fashion_mnist_cnn_paths["logpath"],
    save_dir=run_fashion_mnist_cnn_paths["save_dir"],
    show_sample=True,
    show_filters=True,
)
fashion_mnist_cnn_solver.train()

In [ ]:
run_fashion_mnist_cnn_hyper_paths = build_run_paths("fashion_mnist", "simplecnn", hyper_mode=True)

fashion_mnist_cnn_hyper_solver = Solver(
    dataset="fashion_mnist",
    model="simplecnn",
    max_epoch=1,
    hyper_mode=True,
    logpath=run_fashion_mnist_cnn_hyper_paths["logpath"],
    save_dir=run_fashion_mnist_cnn_hyper_paths["save_dir"],
    show_filters=True,
)
fashion_mnist_cnn_hyper_solver.train()

In [ ]:
run_cifar10_resnet50_paths = build_run_paths("cifar10", "resnet50")

cifar10_resnet50_solver = Solver(
    dataset="cifar10",
    model="resnet50",
    max_epoch=10,
    hyper_mode=False,
    logpath=run_cifar10_resnet50_paths["logpath"],
    save_dir=run_cifar10_resnet50_paths["save_dir"],
    show_sample=True,
    show_filters=True,
)
cifar10_resnet50_solver.train()

In [ ]:
run_cifar10_resnet50_hyper_paths = build_run_paths("cifar10", "resnet50", hyper_mode=True)

cifar10_resnet50_hyper_solver = Solver(
    dataset="cifar10",
    model="resnet50",
    max_epoch=10,
    hyper_mode=True,
    logpath=run_cifar10_resnet50_hyper_paths["logpath"],
    save_dir=run_cifar10_resnet50_hyper_paths["save_dir"],
    show_sample=True,
    show_filters=True,
)
cifar10_resnet50_hyper_solver.train()

In [ ]:
run_svhn_resnet50_paths = build_run_paths("svhn", "resnet50")

svhn_resnet50_solver = Solver(
    dataset="svhn",
    model="resnet50",
    max_epoch=10,
    hyper_mode=False,
    logpath=run_svhn_resnet50_paths["logpath"],
    save_dir=run_svhn_resnet50_paths["save_dir"],
    show_sample=True,
    show_filters=True,
)
svhn_resnet50_solver.train()

In [ ]:
run_svhn_resnet50_hyper_paths = build_run_paths("svhn", "resnet50", hyper_mode=True)

svhn_resnet50_hyper_solver = Solver(
    dataset="svhn",
    model="resnet50",
    max_epoch=10,
    hyper_mode=True,
    logpath=run_svhn_resnet50_hyper_paths["logpath"],
    save_dir=run_svhn_resnet50_hyper_paths["save_dir"],
    show_sample=True,
    show_filters=True,
)
svhn_resnet50_hyper_solver.train()

In [ ]:

print("Đang tải dữ liệu và nạp mô hình...")

# 1. Tải dữ liệu test MNIST
dataset = Mnist()
x_test = dataset.x_test
y_test = dataset.y_test

# 2. Khởi tạo lại mô hình (Phải giống hệt với lúc bạn cấu hình train)
# Nếu lúc train bạn dùng --hyper-mode, hãy đổi hyper_mode=True ở đây
model = SimpleCNN(num_classes=10, hyper_mode=True)

# Chạy thử 1 batch rỗng để mô hình tự khởi tạo kích thước các lớp (build model)
dummy_input = tf.zeros((1, 28, 28, 1), dtype=tf.float32)
model(dummy_input, training=False)

# 3. Nạp trọng số đã train (Checkpoint)
# Lưu ý: Cập nhật lại đường dẫn này cho khớp với thư mục lưu của bạn.
# Dựa vào log của bạn ở trên, đường dẫn có vẻ là: 'runs/mnist_simplecnn/checkpoints/latest'
# Nếu chạy mặc định thì nó là 'checkpoints/latest' hoặc 'checkpoints/best'
checkpoint_dir = 'runs/mnist_simplecnn_hyper/checkpoints/best' 

checkpoint = tf.train.Checkpoint(model=model)
manager = tf.train.CheckpointManager(checkpoint, directory=checkpoint_dir, max_to_keep=1)

if manager.latest_checkpoint:
    checkpoint.restore(manager.latest_checkpoint).expect_partial()
    print(f"✅ Đã nạp thành công trọng số từ: {manager.latest_checkpoint}")
else:
    print("❌ Không tìm thấy checkpoint! Hãy kiểm tra lại đường dẫn biến checkpoint_dir.")

# 4. Chọn ngẫu nhiên 5 bức ảnh để kiểm tra
num_samples = 5
indices = random.sample(range(len(x_test)), num_samples)

plt.figure(figsize=(15, 4))
for i, idx in enumerate(indices):
    image = x_test[idx]
    true_label = np.argmax(y_test[idx]) # Chuyển nhãn (one-hot) về số nguyên

    # 5. Dự đoán
    # Thêm chiều batch ở đầu (28, 28, 1) -> (1, 28, 28, 1) để đưa vào mạng
    image_batch = np.expand_dims(image, axis=0) 
    logits = model(image_batch, training=False)
    predicted_label = np.argmax(tf.nn.softmax(logits).numpy()) # Lấy class có xác suất cao nhất

    # 6. Hiển thị ảnh kèm theo nhãn
    plt.subplot(1, num_samples, i + 1)
    plt.imshow(np.reshape(image, (28, 28)), cmap='Greys', interpolation='nearest')
    
    # Đổi màu text: Xanh nếu đoán ĐÚNG, Đỏ nếu đoán SAI
    color = 'green' if predicted_label == true_label else 'red'
    plt.title(f"Dự đoán: {predicted_label}\nThực tế: {true_label}", color=color, fontweight='bold')
    plt.axis('off')

plt.tight_layout()
plt.show()